# Milestone 3 lab, notebook 5 of 8: A file you do not control

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| **5. A file you do not control** (you are here) | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
import pandas as pd

from lab_helpers import LANDING, keep, lab_copy, on_lab, plant, prepare_week, production, show_card, sql, standard_checks

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: An order with no cost

**Finance** approves tomorrow's order every morning. They look at one number: what will it cost?

This is the order list the pipeline made on the night of Tuesday 25 August, for Wednesday.

In [2]:
sql("""
    SELECT count(*)                          AS rows_on_the_list,
           count(*) - count(unit_cost_cents) AS rows_with_no_cost,
           sum(CASE WHEN unit_cost_cents IS NULL THEN order_qty ELSE 0 END) AS units_ordered_with_no_cost
    FROM output.replenishment_list
    WHERE for_date = DATE '2026-08-26'
""")

,rows_on_the_list,rows_with_no_cost,units_ordered_with_no_cost
0,234,72,154.895


72 rows of the list have no cost: twelve products in six stores. Finance cannot approve them, and cannot pay for them. The day before, every row had a cost.

Which twelve products?

In [3]:
sql("""
    SELECT s.supplier_id, count(DISTINCT r.product_id) AS products_with_no_cost
    FROM output.replenishment_list r
    JOIN clean.supplier_prices s ON s.product_id = r.product_id AND s.price_date = DATE '2026-08-25'
    WHERE r.for_date = DATE '2026-08-26' AND r.unit_cost_cents IS NULL
    GROUP BY s.supplier_id
""")

,supplier_id,products_with_no_cost
0,SUP-B,12


All twelve come from one supplier, `SUP-B`, and its file of 25 August.

## Step 2: The card. Who makes this data?

In [4]:
show_card("supplier_prices", 3)

**From the check card `checks/supplier_prices.md`, question 3. How is the data made? So what can go wrong?**

**Lotus does not make this data, and does not control it.** Another company's program writes the file. That company can change the program on any day, and nobody at Lotus is told.

| Step | Made by | What can go wrong |
|---|---|---|
| The supplier's system writes a CSV file | A program Lotus has never seen | A column is renamed, moved or removed. A column is added |
| The cost is written | The same program | The unit of the number changes: cents where dollars were agreed |
| The file is sent once a week | The supplier | The same file again. No file |

Read the first sentence again. Every table so far was made inside Lotus: by a till, a buyer, a store worker, a customer in Lotus's own app. If one of those goes wrong, someone at Lotus can walk over and ask.

This file is written by another company's program. They can change that program on any day. Nobody at Lotus is told.

Look at what was loaded from `SUP-B`'s last two files.

In [5]:
sql("""
    SELECT _file_name, count(*) AS rows, count(unit_cost) AS rows_with_a_cost
    FROM raw.supplier_prices
    WHERE supplier_id = 'SUP-B' AND price_date >= '2026-08-18'
    GROUP BY _file_name
    ORDER BY _file_name
""")

,_file_name,rows,rows_with_a_cost
0,supplier_prices_SUP-B_2026-08-18.csv,12,12
1,supplier_prices_SUP-B_2026-08-25.csv,12,0


Twelve rows each. On 18 August, twelve costs. On 25 August, none.

The table cannot tell you more. So open the two files themselves, as they arrived.

In [6]:
folder = LANDING / "supplier_prices"
for name in ("supplier_prices_SUP-B_2026-08-18.csv", "supplier_prices_SUP-B_2026-08-25.csv"):
    print(name)
    print(pd.read_csv(folder / name, dtype=str).head(3).to_string(index=False))
    print()

supplier_prices_SUP-B_2026-08-18.csv
supplier_id product_id unit_cost unit price_date
      SUP-B      P-012      0.89   kg 2026-08-18
      SUP-B      P-013      2.19   kg 2026-08-18
      SUP-B      P-014      2.27   kg 2026-08-18

supplier_prices_SUP-B_2026-08-25.csv
supplier_id product_id cost_per_unit unit price_date
      SUP-B      P-012          0.89   kg 2026-08-25
      SUP-B      P-013          2.19   kg 2026-08-25
      SUP-B      P-014          2.27   kg 2026-08-25



There it is, in the first line. The column `unit_cost` is now called `cost_per_unit`. The costs are all there, under a new name.

The loader reads each file by column **name**. It asked for `unit_cost`, found no such column, and wrote an empty cell. Twelve times. It did not fail.

The five standard checks?

In [7]:
standard_checks()

1. The table is not empty           0 bad rows  PASS
2. The key is not null              0 bad rows  PASS
3. The key is unique                0 bad rows  PASS
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


Green. They look at `sales_lines`. They have never heard of this file.

## Step 3: Its name

The column names of a file, the type of each column, and the rules for its values are called its **schema**. It is the shape of the file.

What `SUP-B` did is called **schema drift**: a schema that changes without anyone agreeing.

Here is the problem with checking it. Look at the raw table's own columns.

In [8]:
sql("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'supplier_prices'
    ORDER BY ordinal_position
""")

,column_name
0,supplier_id
1,product_id
2,unit_cost
3,unit
4,price_date
5,_file_name
6,_received_at
7,_loaded_at


The table has a column `unit_cost`, and no column `cost_per_unit`. The table has the shape the **loader** expected. It cannot tell you the shape the **file** had. A column the loader did not expect is simply not there.

So from this milestone on, the loader keeps the first line of every file: one row per column, in `raw.file_columns`.

In [9]:
sql("""
    SELECT file_name, position_in_file, column_name
    FROM raw.file_columns
    WHERE file_name IN ('supplier_prices_SUP-B_2026-08-18.csv', 'supplier_prices_SUP-B_2026-08-25.csv')
    ORDER BY file_name, position_in_file
""")

,file_name,position_in_file,column_name
0,supplier_prices_SUP-B_2026-08-18.csv,1,supplier_id
1,supplier_prices_SUP-B_2026-08-18.csv,2,product_id
2,supplier_prices_SUP-B_2026-08-18.csv,3,unit_cost
3,supplier_prices_SUP-B_2026-08-18.csv,4,unit
4,supplier_prices_SUP-B_2026-08-18.csv,5,price_date
5,supplier_prices_SUP-B_2026-08-25.csv,1,supplier_id
6,supplier_prices_SUP-B_2026-08-25.csv,2,product_id
7,supplier_prices_SUP-B_2026-08-25.csv,3,cost_per_unit
8,supplier_prices_SUP-B_2026-08-25.csv,4,unit
9,supplier_prices_SUP-B_2026-08-25.csv,5,price_date


## Step 4: Write the check. Expected against found

Put the columns you expect beside the columns each file really had. Two things can be wrong, and they are not equally bad.

- An expected column is **missing**. Whatever reads it gets nothing.
- A column is there that nobody **expected**. Nothing reads it, so nothing breaks.

In [10]:
EXPECTED = """
    SELECT 'supplier_id' AS column_name UNION ALL SELECT 'product_id' UNION ALL
    SELECT 'unit_cost' UNION ALL SELECT 'unit' UNION ALL SELECT 'price_date'
"""
FILES_HAVE_THEIR_COLUMNS = f"""
    SELECT f.file_name AS thing, e.column_name,
           'an expected column is missing' AS problem, 'breaking' AS kind_of_change
    FROM (SELECT DISTINCT file_name FROM raw.file_columns WHERE table_name = 'supplier_prices') f
    CROSS JOIN ({EXPECTED}) e
    LEFT JOIN raw.file_columns found
      ON found.file_name = f.file_name AND found.column_name = e.column_name
    WHERE found.column_name IS NULL
    UNION ALL
    SELECT found.file_name AS thing, found.column_name,
           'a column we did not expect' AS problem, 'additive' AS kind_of_change
    FROM raw.file_columns found
    LEFT JOIN ({EXPECTED}) e ON e.column_name = found.column_name
    WHERE found.table_name = 'supplier_prices' AND e.column_name IS NULL
"""
yours = sql(FILES_HAVE_THEIR_COLUMNS)
yours

,thing,column_name,problem,kind_of_change
0,supplier_prices_SUP-B_2026-08-25.csv,unit_cost,an expected column is missing,breaking
1,supplier_prices_SUP-B_2026-08-25.csv,cost_per_unit,a column we did not expect,additive


Two rows for one file. A rename is two changes at once: one column went missing, and one new column appeared.

Look at the last column. The two kinds of change have names.

- A **breaking change** removes or changes something a reader uses, so old readers fail. Here the reader did not even fail loudly. It read nothing.
- An **additive change** only adds. Old readers still work.

The production check does the same for the files of all seven tables. Only this one file has a finding.

In [11]:
keep("files_have_their_columns", yours)

Your query:           2 bad rows
The production check: 2 bad rows  (A file has the columns we expect)
The same rows.


See an additive change alone. Suppose the loyalty app starts sending one more column, `nickname`.

In [12]:
lab_copy("raw.file_columns")
plant("INSERT INTO lab.raw_file_columns VALUES ('members_2026-08-30.csv', 'members', 'nickname', 8)")
production("files_have_their_columns", on_the_lab=True)[["thing", "column_name", "kind_of_change"]]

lab.raw_file_columns is a fresh copy of raw.file_columns: 6,488 rows.


Planted in the lab copy.


,thing,column_name,kind_of_change
0,supplier_prices_SUP-B_2026-08-25.csv,unit_cost,breaking
1,supplier_prices_SUP-B_2026-08-25.csv,cost_per_unit,additive
2,members_2026-08-30.csv,nickname,additive


The new column is reported as additive. Nothing broke: the loader never asks for `nickname`. It is still worth one line in a report. It tells you the sender changed something, and the next change may not be harmless.

## Step 5: The check that speaks later

There is a second check that finds this fault. It is the not-null check from milestone 2: every supplier row has a cost.

In [13]:
COST_IS_FILLED = """
    SELECT supplier_id || ' ' || product_id AS thing, price_date AS day,
           supplier_id, product_id, unit_cost_cents
    FROM clean.supplier_prices
    WHERE unit_cost_cents IS NULL OR product_id IS NULL OR supplier_id IS NULL
       OR unit IS NULL OR price_date IS NULL
"""
yours = sql(COST_IS_FILLED)
print(len(yours), "rows with no cost")
yours.head(3)

12 rows with no cost


,thing,day,supplier_id,product_id,unit_cost_cents
0,SUP-B P-012,2026-08-25,SUP-B,P-012,None
1,SUP-B P-013,2026-08-25,SUP-B,P-013,None
2,SUP-B P-014,2026-08-25,SUP-B,P-014,None


In [14]:
keep("cost_is_filled", yours)

Your query:           12 bad rows
The production check: 12 bad rows  (Every supplier row has a cost)
The same rows.


Twelve rows. Both checks catch the same fault. They do not say the same thing.

| | Every supplier row has a cost | A file has the columns we expect |
|---|---|---|
| It looks at | The table, after the load | The first line of the file |
| It says | Twelve costs are empty | `unit_cost` is missing, and `cost_per_unit` is new |
| Who can act on it | Nobody yet. Empty why? | Anyone: the supplier renamed a column |

The second one names the cause. Keep both: the first also catches a file where the column is there and the cells are empty.

## What you now know

- A **schema** is the shape of a table or a file: the column names, the types, and the rules for the values.
- **Schema drift** is a schema that changes without anyone agreeing.
- A **breaking change** removes or changes something a reader uses. An additive change only adds.
- A loaded table has the shape the loader expected. To check the shape of a file, keep its first line.

**Say it in an interview:** "A supplier renamed a cost column. Our loader read by column name, wrote twelve empty costs and did not fail, and the order list went to finance with no cost on 72 rows. A loaded table cannot tell you the shape the file had, so I keep each file's header and compare it with the columns we expect. A missing column is a breaking change. A new one is additive, and I report it."

**The theory:** [Schema Evolution & Data Contracts](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/schema-evolution-and-data-contracts), sections "What is a schema?", "Safe and breaking changes" and "Schema checks in the job".

Next: [notebook 6](06_same_shape_new_meaning.ipynb). The next file has every column in its place. It is far worse.